# 03 — Uncertainty

**Phenomenological Growth Models Lab**

A point estimate on its own is close to useless. "Final epidemic size 7325" invites a
confidence it has not earned. This notebook puts intervals around the parameters and the
predictions, and then scores those intervals honestly.

The punchline you should carry away: **a model can have the lowest error and still be
the worse forecast.** By the end you will have seen that happen.

Assumes [01](01_fitting_basics.ipynb) and [02](02_model_selection.ipynb).

## Setup

These notebooks run against the environment pinned in this repository's `Project.toml`
and `Manifest.toml`. The cell below activates it and installs anything missing.

The first run downloads and precompiles the dependencies — a few minutes, once.
Every run after that starts in seconds.

Because the versions are pinned in `Manifest.toml`, you get the same package versions
these notebooks were written against, whatever else is installed on your machine. If you
skip this cell and rely on your global environment, newer versions may resolve
differently and the numbers below may not match.

In [ ]:
using Pkg

# Works whether the notebook was launched from the repository root or from notebooks/.
root = isfile("Project.toml") ? "." : ".."
Pkg.activate(root)
Pkg.instantiate()

println("Active project: ", Base.active_project())

In [ ]:
using GrowthFit
using Plots, Statistics, Printf, DelimitedFiles, Random

default(size = (760, 420), lw = 2, legend = :topright, framestyle = :box,
        guidefontsize = 10, titlefontsize = 11)

# Threads. Notebooks 03 and 05 bootstrap in parallel. A Jupyter kernel does
# NOT inherit `-t` from whatever launched Jupyter — the thread count is fixed
# by the kernel itself — so if this says 1, see the README for the one-off
# `IJulia.installkernel` step. Results are identical either way; only speed
# changes.
println("GrowthFit loaded. Threads available: ", Threads.nthreads())
Threads.nthreads() == 1 && println("  (single-threaded — notebooks 03 and 05 will be slower; see README)")


> **Tip.** The bootstrap in notebooks 03 and 05 runs in parallel. If you want it faster,
> restart Julia with `-t 8`. Results are identical either way — each replicate is seeded
> independently of the thread count.

In [ ]:
function load_series(name)
    path = isfile(joinpath("data", name)) ? joinpath("data", name) :
           joinpath("..", "data", name)
    isfile(path) || error("Could not find $name. Run this notebook from inside the " *
                          "cloned repository so the data/ folder is reachable.")
    raw = readdlm(path, '\t', Float64)
    return raw[:, 1], raw[:, 2]          # time, incidence
end

timevect_all, ydata_all = load_series("jalisco_measles.txt")

@printf("%d weeks, peak of %.0f cases, %.0f cases total\n",
        length(ydata_all), maximum(ydata_all), sum(ydata_all))

In [ ]:
calibration_period = 40
forecast_horizon   = 3
timevect = timevect_all[1:calibration_period]
ydata    = ydata_all[1:calibration_period]
I0 = ydata[1]

## Parametric bootstrap

The recipe:

1. Fit the model to the real data. Call the fitted curve $\hat{\mu}_t$.
2. Generate a synthetic dataset by drawing each week's count from the assumed error
   distribution centred on $\hat{\mu}_t$ — here negative binomial with the fitted $\alpha$.
3. Refit the model to that synthetic dataset. Record the parameters.
4. Repeat $M$ times.

The spread of those $M$ parameter sets is the sampling uncertainty. It answers: *if the
world really worked like my fitted model, how much would my estimates bounce around?*

Note what that does **not** include. It assumes the model is correct. If the true curve
is not a Generalized Richards Model, no amount of bootstrapping will tell you.

First, the point fit:

In [ ]:
Random.seed!(1)
fit_grm = fit_growth_model(:grm, timevect, ydata; dist = :nb1, n_restarts = 10)
@printf("GRM: r=%.4f p=%.4f a=%.4f K=%.0f alpha=%.2f\n",
        fit_grm.r, fit_grm.p_exp, fit_grm.a, fit_grm.K, fit_grm.alpha)

Now the bootstrap. `M = 300` replicates is a reasonable default; use fewer while you
are experimenting.

The `seed` argument makes this reproducible — and reproducible *regardless of how many
threads you have*, which is not automatic in parallel code. Always pass one when you
report an interval.

In [ ]:
boot_grm = run_bootstrap(:grm, timevect, ydata, fit_grm;
                         dist = :nb1, M = 300,
                         forecast_horizon = forecast_horizon,
                         seed = 20260726)

println("successful replicates: $(boot_grm.n_success) / 300")

If that count is well below `M`, replicates are failing to converge and the intervals
will be too narrow — the failures are not random, they cluster in awkward regions of
parameter space.

## Parameter distributions

`boot.params` is an `M × 5` matrix: columns are $r$, $p$, $a$, $K$, $\alpha$.

In [ ]:
pnames = ["r", "p", "a", "K", "alpha"]

ci95(v) = (median(v), quantile(v, 0.025), quantile(v, 0.975))

println("parameter      median      2.5%       97.5%     width/median")
println("-"^62)
for (i, nm) in enumerate(pnames)
    m, lo, hi = ci95(boot_grm.params[:, i])
    @printf("%-10s %10.4f %10.4f %10.4f %10.2f\n", nm, m, lo, hi, (hi - lo) / abs(m))
end

That last column — interval width relative to the estimate — is the quick diagnostic.
Anything above about 1 means the interval is as wide as the estimate itself, and you
should be wary of interpreting that parameter on its own.

Look at which parameters are tight and which are not. $K$ is usually reasonable; $a$
usually is not. Notebook 04 explains why.

In [ ]:
plts = [histogram(boot_grm.params[:, i], bins = 30, label = "",
                  title = pnames[i], lc = :white, fc = :steelblue,
                  titlefontsize = 10)
        for i in 1:4]
plot(plts..., layout = (2, 2), size = (820, 480))

## Prediction intervals

Parameter uncertainty is not the whole story. Even with the parameters known exactly,
next week's count would not land on the curve — there is observation noise on top.

`boot.forecast_noisy` carries both: each column is one replicate's trajectory with
observation noise applied. Quantiles across columns give a genuine **prediction
interval** for what you would actually observe.

In [ ]:
nT = size(boot_grm.forecast_noisy, 1)
tfull = timevect_all[1:nT]

med = [median(boot_grm.forecast_noisy[i, :]) for i in 1:nT]
lo  = [quantile(boot_grm.forecast_noisy[i, :], 0.025) for i in 1:nT]
hi  = [quantile(boot_grm.forecast_noisy[i, :], 0.975) for i in 1:nT]

plot(tfull, med, ribbon = (med .- lo, hi .- med), fillalpha = 0.25,
     label = "median + 95% PI", lc = :crimson, fc = :crimson,
     xlabel = "week", ylabel = "weekly cases", title = "GRM with prediction interval")
scatter!(timevect, ydata, label = "calibration data", mc = :steelblue, ms = 4, msw = 0)
scatter!(timevect_all[calibration_period+1:nT], ydata_all[calibration_period+1:nT],
         label = "held out", mc = :orange, ms = 5, msw = 0, marker = :diamond)
vline!([timevect[end] + 0.5], label = "", ls = :dash, lc = :grey)

## Scoring: four metrics that measure different things

We will compute these by hand. They are simple, and seeing the formulas is the point.

**MAE** — mean absolute error of the point prediction:
$$\mathrm{MAE} = \frac{1}{n}\sum_t |y_t - \hat{y}_t|$$

**RMSE** — root mean squared error. Squaring first means large misses dominate:
$$\mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_t (y_t - \hat{y}_t)^2}$$

**Coverage** — the fraction of observations that fell inside the 95% interval. Nothing
to do with the point estimate. If your intervals are honest this should be near 0.95.

**WIS** — the weighted interval score. Rewards accuracy *and* sharpness together:
$$\mathrm{WIS} = \frac{1}{K + 1/2}\left(\tfrac{1}{2}|y - m| + \sum_{k=1}^{K} \tfrac{\alpha_k}{2} \mathrm{IS}_{\alpha_k}(y)\right)$$

where the interval score for a central $(1-\alpha)$ interval $[l, u]$ is

$$\mathrm{IS}_\alpha(y) = (u - l) + \tfrac{2}{\alpha}(l - y)\mathbb{1}\{y < l\} + \tfrac{2}{\alpha}(y - u)\mathbb{1}\{y > u\}$$

Read that as: pay for the width of your interval, plus a penalty if the truth falls
outside it. Narrow intervals are cheap *when they are right* and expensive when they
are not. Lower is better, and WIS is on the same scale as the data — cases, here.

In [ ]:
function interval_score(y, l, u, α)
    s = (u - l)
    y < l && (s += 2 / α * (l - y))
    y > u && (s += 2 / α * (y - u))
    return s
end

"""
Weighted interval score for one observation against a sample of predictions.

The α levels are 0.02, 0.05, and 0.1 through 0.9 — eleven of them, matching
the forecast hubs and `computeWIS.m` in the R and MATLAB toolboxes. The set
matters: WIS is an average over these levels, so a shorter list gives a
different (not wrong, just different) number.
"""
function wis(y, sample; αs = vcat(0.02, 0.05, collect(0.1:0.1:0.9)))
    m = median(sample)
    K = length(αs)
    total = 0.5 * abs(y - m)
    for α in αs
        l = quantile(sample, α / 2)
        u = quantile(sample, 1 - α / 2)
        total += (α / 2) * interval_score(y, l, u, α)
    end
    return total / (K + 0.5)
end

"""
All four metrics over the calibration period.

Note which array each uses. MAE and RMSE score the *point forecast*, so they
use `fit_curves` — the fitted model trajectories. Coverage and WIS are about
the *predictive distribution*, so they use `forecast_noisy`, which adds
observation noise on top. Scoring MAE against the noisy curves instead would
be measuring the model against a random realization of itself.
"""
function score_calibration(boot, ydata)
    n = length(ydata)
    fit   = [boot.fit_curves[i, :] for i in 1:n]        # no observation noise
    preds = [boot.forecast_noisy[i, :] for i in 1:n]    # with observation noise

    m  = [median(f) for f in fit]
    lo = [quantile(p, 0.025) for p in preds]
    hi = [quantile(p, 0.975) for p in preds]

    return (mae      = mean(abs.(ydata .- m)),
            rmse     = sqrt(mean((ydata .- m) .^ 2)),
            coverage = mean(lo .<= ydata .<= hi),
            wis      = mean(wis(ydata[i], preds[i]) for i in 1:n))
end

s = score_calibration(boot_grm, ydata)
@printf("MAE      %8.2f cases\n", s.mae)
@printf("RMSE     %8.2f cases\n", s.rmse)
@printf("Coverage %8.1f %%\n", 100 * s.coverage)
@printf("WIS      %8.2f\n", s.wis)

The package computes the same things. Because we matched its choices — the
point forecast from `fit_curves`, the eleven-level α set — the numbers should
agree. Note `MSE` rather than `RMSE`; take the square root to compare.

In [ ]:
calib = compute_calibration_performance(boot_grm, ydata)
println("Fields returned by compute_calibration_performance:")
println(propertynames(calib))
calib

### Reading the numbers

**MAE vs RMSE.** RMSE will be larger — it always is, unless every error is identical.
The size of the gap tells you how skewed the errors are. On an epidemic curve, a big
gap usually means the model is missing the peak: a handful of large errors near the
maximum dominate the squared term while barely moving the absolute one.

**Coverage.** If this is well below 95%, the model is overconfident. That is common,
and it usually means the parametric bootstrap has understated uncertainty — it assumes
the model form is right, and it is not.

## The comparison that matters

Now score two models against each other: the Generalized Richards Model, and the
Generalized Growth Model — which has no carrying capacity and therefore cannot turn over.

In [ ]:
Random.seed!(1)
fit_ggm  = fit_growth_model(:ggm, timevect, ydata; dist = :nb1, n_restarts = 10)
boot_ggm = run_bootstrap(:ggm, timevect, ydata, fit_ggm;
                         dist = :nb1, M = 300,
                         forecast_horizon = forecast_horizon, seed = 20260726)

Random.seed!(1)
fit_glm  = fit_growth_model(:glm, timevect, ydata; dist = :nb1, n_restarts = 10)
boot_glm = run_bootstrap(:glm, timevect, ydata, fit_glm;
                         dist = :nb1, M = 300,
                         forecast_horizon = forecast_horizon, seed = 20260726)

results = Dict(:grm => score_calibration(boot_grm, ydata),
               :glm => score_calibration(boot_glm, ydata),
               :ggm => score_calibration(boot_ggm, ydata))

println("model    MAE      RMSE     Coverage    WIS")
println("-"^48)
for f in (:grm, :glm, :ggm)
    r = results[f]
    @printf("%-8s %7.2f %8.2f %9.1f%% %7.2f\n", f, r.mae, r.rmse, 100r.coverage, r.wis)
end

Look for a model that does well on MAE while doing badly on coverage and WIS. That is
the pattern to internalize: **the point prediction can be close while the uncertainty
is dishonest.** MAE cannot see that. WIS can, because it charges for interval width and
penalizes misses outside the interval.

Plot the intervals side by side and the difference becomes visible:

In [ ]:
function band(boot, label, colour)
    nT = size(boot.forecast_noisy, 1)
    m  = [median(boot.forecast_noisy[i, :]) for i in 1:nT]
    l  = [quantile(boot.forecast_noisy[i, :], 0.025) for i in 1:nT]
    u  = [quantile(boot.forecast_noisy[i, :], 0.975) for i in 1:nT]
    return timevect_all[1:nT], m, l, u, label, colour
end

plt = plot(xlabel = "week", ylabel = "weekly cases",
           title = "95% prediction intervals compared", legend = :topleft)
for (t, m, l, u, lab, c) in (band(boot_grm, "GRM", :crimson),
                             band(boot_glm, "GLM", :seagreen),
                             band(boot_ggm, "GGM", :purple))
    plot!(plt, t, m, ribbon = (m .- l, u .- m), fillalpha = 0.15,
          label = lab, lc = c, fc = c)
end
scatter!(plt, timevect, ydata, label = "observed", mc = :black, ms = 3, msw = 0)
plt

---

## Exercises

1. Re-run the GRM bootstrap with `dist = :poisson` instead of `:nb1`. What happens to
   coverage, and why? (Poisson forces $\mathrm{Var} = \mu$; the data are far more
   variable than that.)

2. Drop to `M = 30`. How much do the interval endpoints move if you change the seed?
   That variability is Monte Carlo error, and it tells you whether $M$ is large enough.

3. Cut the α set in `wis` down to `[0.02, 0.05, 0.1, 0.2, 0.5]` and re-score. How much
   does WIS move? This is why two papers reporting "WIS" are not always comparable.

4. Compute the metrics separately for the first 20 weeks and the last 20. Where is the
   model doing badly — the growth phase or the decline?

**Next:** [04 — Identifiability](04_identifiability.ipynb), which explains why the
interval on $a$ was so wide.